In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Credentials come from the encrypted Databricks secret scope `dea-rds`
# (created with the SETUP_secrets notebook at the workspace root).
# No connection values are stored in code or git — they were previously
# hardcoded here and exposed via the public GitHub repo (since rotated).
SECRET_SCOPE = "dea-rds"

jdbc_host = dbutils.secrets.get(scope=SECRET_SCOPE, key="host")
jdbc_port = dbutils.secrets.get(scope=SECRET_SCOPE, key="port")
jdbc_database = dbutils.secrets.get(scope=SECRET_SCOPE, key="database")

jdbc_url = (
    f"jdbc:postgresql://{jdbc_host}:{jdbc_port}/{jdbc_database}"
)

connection_properties = {
    "user": dbutils.secrets.get(scope=SECRET_SCOPE, key="username"),
    "password": dbutils.secrets.get(scope=SECRET_SCOPE, key="password"),
    "driver": "org.postgresql.Driver"
}

CATALOG = "crm_ingestion"
BRONZE_SCHEMA = "bronze"

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{BRONZE_SCHEMA}
""")

print("✅ Connection loaded (credentials from secret scope dea-rds)")

In [0]:
config_df = (
    spark.table("crm_ingestion.config.bronze_config")
         .filter("active = true")
)

display(config_df)

In [0]:
# =====================================================================
# OOM-SAFE LOAD SETTINGS (batching design, Nov 3 2026)
# =====================================================================
import time
# Python's builtin sum/min are SHADOWED in this notebook by cell 1's
# `from pyspark.sql.functions import *` (pyspark exports sum, min, max, …).
# Same trap the utils notebook already worked around with spark_sum.
# Call the REAL builtins explicitly wherever Python semantics are meant.
import builtins

JDBC_FETCHSIZE = 10000      # Stream rows from Postgres. The JDBC driver's
                            # DEFAULT behavior buffers the ENTIRE result set
                            # in driver memory — the lead_activites failure.
BATCH_BUDGET_MB = 256       # Max estimated batch size before splitting
                            # (estimated via pg_stats.avg_width × dump
                            # row counts — see load_postgres_table).
FALLBACK_ROW_BYTES = 1024   # If pg_stats has no stats for a table.
BATCH_PARTITION_MIN_MB = 64 # Batches at least this big are fetched in
                            # parallel via partitionColumn (one JDBC
                            # connection per partition).
BATCH_PARTITIONS = 8        # Parallel JDBC connections per large batch.
BATCH_DELAY_SEC = 10        # Breather between batches — kindness to the
                            # source RDS under large catch-up pulls.


def _jdbc_query_reader(dbtable):
    """Shared JDBC reader: one connection spec, fetchsize ALWAYS on."""
    return (spark.read
                 .format("jdbc")
                 .option("url", jdbc_url)
                 .option("dbtable", dbtable)
                 .option("user", connection_properties["user"])
                 .option("password", connection_properties["password"])
                 .option("driver", connection_properties["driver"])
                 .option("fetchsize", str(JDBC_FETCHSIZE)))


def _pg_avg_row_bytes(source_table):
    """Estimated avg row size (bytes) from Postgres pg_stats.avg_width —
    the sum over columns approximates one row's width. Falls back to
    FALLBACK_ROW_BYTES when the table has never been ANALYZEd."""
    schema, _, table = source_table.rpartition(".")
    if not schema:
        schema = "public"
    try:
        w = (_jdbc_query_reader(f"""
            (SELECT SUM(avg_width) AS w FROM pg_stats
             WHERE schemaname = '{schema}' AND tablename = '{table}') t
        """).load().collect()[0]["w"])
        return int(w) if w else FALLBACK_ROW_BYTES
    except Exception:
        return FALLBACK_ROW_BYTES


def load_postgres_table(
    source_table: str,
    target_table: str,
    incremental_column: str
):

    # No backfill/search mode — just loads (user-directed Nov 3 2026).
    # A missing/empty bronze table triggers the project's FULL LOAD definition:
    # the last 30 days of source dumps. Because the source re-dumps the full
    # current row set every night, a single dump already carries every current
    # record; 30 days also captures recent change versions. Silver's engine
    # dedup distills the copies down to one row per distinct payload version.
    try:
        max_row = spark.sql(f"SELECT MAX({incremental_column}) AS max_date FROM {target_table}").collect()[0]
        max_date = max_row["max_date"] if max_row else None
    except Exception:
        max_date = None

    if max_date:
        # Full microsecond precision. Every row of a source dump shares ONE
        # timestamp (e.g. the Oct 28 dump = all 104,587 rows at 11:01:19.948),
        # so a seconds-truncated watermark (old: str.split('.')[0]) makes the
        # ENTIRE last dump satisfy `> watermark` and re-loads it on every run
        # and every re-run — proven Nov 3 2026 on the other workspace, where
        # an "incremental" run loaded exactly the previous dump's 104,587 rows.
        # PostgreSQL compares microsecond timestamps natively — no stripping.
        max_str = max_date.strftime("%Y-%m-%d %H:%M:%S.%f")
        where_clause = f"{incremental_column} > TIMESTAMP '{max_str}'"
        mode_label = f"incremental (after {max_str})"
    else:
        # FULL LOAD (first run): last 30 days — the project's full-load
        # definition ("load <= 30 days").
        where_clause = f"DATE({incremental_column}) >= (CURRENT_DATE - 30)"
        mode_label = "full load (last 30 days)"

    # ------------------------------------------------------------------
    # MB-BASED BATCHING — the OOM defense (design agreed Nov 3 2026).
    # Every row of a source dump shares ONE timestamp (proven Nov 3 2026:
    # the Oct 28 calendly dump = all 104,587 rows at 11:01:19.948), so
    # enumerating DISTINCT insert_date with counts gives exact dump
    # boundaries + exact row counts in one tiny aggregation. Consecutive
    # dumps are packed into batches under BATCH_BUDGET_MB, estimated via
    # pg_stats.avg_width. Batch boundaries align to real dump boundaries
    # (a batch can never slice a dump), never exceed the memory budget,
    # and fetchsize streams each batch row-by-row instead of the JDBC
    # driver buffering the ENTIRE pull in driver memory.
    # No df.count() pre-scan — the enumeration already counts the rows.
    # ------------------------------------------------------------------
    dumps = (_jdbc_query_reader(f"""
        (SELECT {incremental_column} AS ts, COUNT(*) AS n
         FROM {source_table} WHERE {where_clause}
         GROUP BY 1 ORDER BY 1) t
    """).load().collect())

    total_rows = builtins.sum(r["n"] for r in dumps)
    if total_rows == 0:
        print(f"⏭️  {source_table}: 0 rows ({mode_label}) — skipped")
        return

    row_bytes = _pg_avg_row_bytes(source_table)
    total_mb = total_rows * row_bytes / 2**20

    # Pack consecutive dumps greedily into ≤ BATCH_BUDGET_MB batches.
    # A single dump larger than the budget lands in its own batch and is
    # fetched in parallel via partitionColumn (graceful degradation).
    batches, cur, cur_bytes = [], [], 0
    for r in dumps:
        chunk = r["n"] * row_bytes
        if cur and cur_bytes + chunk > BATCH_BUDGET_MB * 2**20:
            batches.append(cur)
            cur, cur_bytes = [], 0
        cur.append((r["ts"], r["n"]))
        cur_bytes += chunk
    if cur:
        batches.append(cur)
    print(f"  📏 ≈{row_bytes} B/row (pg_stats) → pull ≈ {total_mb:,.0f} MB, {total_rows:,} rows → {len(batches)} batch(es) ≤ {BATCH_BUDGET_MB} MB")

    loaded = 0
    for i, batch in enumerate(batches, 1):
        lo, hi = batch[0][0], batch[-1][0]
        lo_str = lo.strftime("%Y-%m-%d %H:%M:%S.%f")
        hi_str = hi.strftime("%Y-%m-%d %H:%M:%S.%f")
        n_rows = builtins.sum(x[1] for x in batch)
        batch_mb = n_rows * row_bytes / 2**20

        batch_where = (f"{incremental_column} >= TIMESTAMP '{lo_str}' "
                       f"AND {incremental_column} <= TIMESTAMP '{hi_str}'")

        reader = _jdbc_query_reader(f"""
            (SELECT * FROM {source_table} WHERE {batch_where}) t
        """)
        if batch_mb >= BATCH_PARTITION_MIN_MB and lo < hi:
            # Parallel JDBC fetch: one connection per partition, bounds =
            # the batch's dump range (distribute roughly one dump per
            # partition). Small batches stay single-connection.
            reader = (reader
                      .option("partitionColumn", incremental_column)
                      .option("lowerBound", lo_str)
                      .option("upperBound", hi_str)
                      .option("numPartitions", str(builtins.min(BATCH_PARTITIONS, len(batch)))))

        df = reader.load()

        # Stamp bronze_updated_at = load time so Silver CDC (bronze_updated_at > watermark)
        # can see incremental rows. Previously this column stayed NULL on every incremental
        # load, which froze all Silver tables at their last full-load state (Oct 6).
        # select("*", <new col>) — NOT withColumn-in-loop (avoids SCPAP004 plan nesting).
        (df.select("*", current_timestamp().alias("bronze_updated_at"))
           .write
           .format("delta")
           .mode("append")
           .option("mergeSchema", "true")
           .saveAsTable(target_table))

        loaded += n_rows
        print(f"  ✅ batch {i}/{len(batches)}: {lo_str} → {hi_str}: {n_rows:,} rows (~{batch_mb:,.0f} MB)")
        if i < len(batches):
            time.sleep(BATCH_DELAY_SEC)

    print(f"✅ Loaded {source_table}: {loaded:,} rows ({mode_label})")

In [0]:
configs = config_df.collect()

# Error isolation (detachment for bronze): the 9 source tables have no
# interdependency, so one failed JDBC load must NOT stop the remaining tables.
# Load everything possible, summarize, then fail the task so the alert fires.
results = []

for row in configs:
    try:
        load_postgres_table(
            source_table=row.source_table,
            target_table=row.target_table,
            incremental_column=row.incremental_column
        )
        results.append((row.target_table, True, None))
    except Exception as e:
        print(f"❌ {row.target_table}: FAILED — {e}")
        results.append((row.target_table, False, str(e)))

print()
print("=" * 60)
print("📊 Bronze ingestion summary")
print("=" * 60)
failed = [(t, err) for t, ok, err in results if not ok]
for t, ok, _ in results:
    print(f"  {'✅' if ok else '❌'} {t}")
print(f"\n{len(results) - len(failed)}/{len(results)} tables loaded")

if failed:
    # Re-run is safe: watermark = MAX(insert_date) in bronze, so the next run
    # re-pulls exactly the missed rows — no duplicates, no manual recovery.
    print(f"⚠️  FAILED: {[t for t, _ in failed]}")
    raise Exception(f"Bronze ingestion: {len(failed)} table(s) failed — see summary above")

print("✅ Bronze ingestion complete")

In [0]:
%sql
DESCRIBE TABLE crm_ingestion.bronze.lead_activites_raw

In [0]:
%sql
SELECT * FROM crm_ingestion.bronze.lead_activites_raw LIMIT 10